In [1]:
import pandas as pd
import numpy as np
import json
import ast  # to safely convert string to dictionary

from pdfs_to_text import pdfs_downloader, pdfs_preprocessing

# Downloading PDFs

In the first notebooks, we obtained the information about building plans and the links to the respective PDFs. In this notebook, you will see how to use the function that takes as input that metadata and downloads all PDFs.

First, read the metadata:

In [2]:
filename = '../data/raw/geoservices_results/map-data.jsonl'

In [3]:
data = []
buffer = ""
with open(filename, 'r', encoding='utf-8') as file:
    for line in file:
        line = line.strip()
        if line:  # Add line to buffer
            buffer += line
            # Check if this is the end of a JSON object
            if line.endswith("}"):
                try:
                    data.append(json.loads(buffer))
                    buffer = ""  # Reset buffer after successful parse
                except json.JSONDecodeError as e:
                    print(f"Error decoding JSON object: {buffer}")
                    print(f"Error: {e}")
                    buffer = ""  # Reset buffer to skip the problematic object


In [4]:
data = pd.DataFrame(data)

In [5]:
data['Easting'] = data['BoundingBox'].apply(lambda x: x['Easting'])
data['Northing'] = data['BoundingBox'].apply(lambda x: x['Northing'])

In [6]:
import geopandas as gpd
from shapely.geometry import Point

In [20]:

# Convert to a GeoDataFrame
data['geometry'] = data.apply(lambda row: Point(row['Easting'], row['Northing']), axis=1)
gdf = gpd.GeoDataFrame(data, geometry='geometry', crs=3857)

In [22]:
high_risk_flooding = gpd.read_file('../data/raw/flood_data/midrisk.shp')

In [23]:
gdf.geometry

0        POINT (1008677.678 6459288.815)
1         POINT (1010502.612 6459298.37)
2        POINT (1015146.161 6459327.034)
3        POINT (1008314.602 6460435.371)
4        POINT (1009078.972 6460483.144)
                      ...               
27487     POINT (1322830.586 6463162.29)
27488    POINT (1322821.031 6463850.223)
27489    POINT (1322696.821 6464413.946)
27490    POINT (1323633.175 6464719.695)
27491    POINT (1322496.174 6464796.132)
Name: geometry, Length: 27492, dtype: geometry

In [24]:
high_risk_flooding.geometry

0     MULTIPOLYGON (((1418650.805 6172770.867, 14186...
1     MULTIPOLYGON (((1447450.373 6045248.65, 144744...
2     MULTIPOLYGON (((1083129.266 6029844.639, 10831...
3     MULTIPOLYGON (((1292920.281 6424048.087, 12929...
4     MULTIPOLYGON (((1287554.801 6438232.799, 12875...
                            ...                        
71    POLYGON ((1268607.723 6536622.307, 1268578.445...
72    POLYGON ((1268605.162 6536569.314, 1268597.593...
73    MULTIPOLYGON (((1102035.448 6050844.638, 11020...
74    MULTIPOLYGON (((1101307.181 6051002.293, 11013...
75    MULTIPOLYGON (((1108629.973 6049195.809, 11086...
Name: geometry, Length: 76, dtype: geometry

In [25]:
data_hr = gdf.sjoin(high_risk_flooding, how="left")

In [28]:
data_hr['flooding_risk'] = data_hr['geometry'].apply(lambda x: 'has_flooding_risk' if not pd.isnull(x) else 'no_flood_risk')

In [29]:
data_hr['flooding_risk'].value_counts()

flooding_risk
has_flooding_risk    27508
Name: count, dtype: int64